# OTP-FM: Beijing Air Quality Forecasting

This notebook demonstrates OTP-FM for modeling temporal evolution of PM2.5 concentrations in Beijing.

The dataset contains 1D PM2.5 measurements from the Dingling station across 13 months.

In [ ]:
import torch
import numpy as np
from collections import OrderedDict
from pathlib import Path

# Import OTP-FM
from otpfm import OTPFM
from otpfm.potentials import W2InfPotential

# Import experiment utilities
from experiments.beijingair.data import load_beijing_data, create_beijing_dataloaders
from experiments.beijingair import BeijingTrainer

## 1. Load Data

In [ ]:
# Load data (downloads automatically)
data = load_beijing_data(
    data_dir=Path("data/beijing"),
    station="Dingling",
    normalize=True,
    ot_coupling=True,
)

print(f"Number of time points: {len(data['marginals'])}")
print(f"Training times: {data['train_times']}")
print(f"Holdout times: {data['holdout_times']}")
for t, m in data['marginals'].items():
    print(f"  Time {t}: {len(m)} measurements")

## 2. Visualize PM2.5 Distributions

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 5))

all_times = sorted(data['marginals'].keys())
pm25_data = [data['marginals'][t].numpy().flatten() for t in all_times]

parts = ax.violinplot(pm25_data, positions=all_times, showmeans=True)

# Color by train/holdout
for i, t in enumerate(all_times):
    color = 'green' if t in data['train_times'] else 'orange'
    if 'bodies' in parts:
        parts['bodies'][i].set_facecolor(color)
        parts['bodies'][i].set_alpha(0.6)

ax.set_xlabel('Time (months)')
ax.set_ylabel('PM2.5 (normalized)')
ax.set_title('PM2.5 Distributions (green=train, orange=holdout)')
plt.show()

## 3. Create Model and Train

In [ ]:
# Create dataloaders
train_loader, val_loader = create_beijing_dataloaders(
    marginals=data['marginals_list'],
    batch_size=128,
    holdout_times=data['holdout_times'],
    ot_alignments=data['ot_alignments'],
)

# Define intermediate times
train_times = data['train_times']
tks = [(t - min(train_times)) / (max(train_times) - min(train_times)) 
       for t in train_times[1:-1]]

# Create potentials
potentials = OrderedDict()
for tk in tks:
    potentials[tk] = W2InfPotential(
        tk=tk,
        strength=100.0,
        lambda_fn_type='gaussian',
        width=0.2,
    )

# Create model (1D data)
model = OTPFM(
    d=1,
    tks=tks,
    potentials=potentials,
    flownet_args={
        'hidden_dim': 64,
        'num_hidden_layers': 2,
    }
)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Train
trainer = BeijingTrainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    save_dir=Path("runs/beijing_demo"),
    lr=1e-3,
    epochs=100,
    potentials=potentials,
    marginals=data['marginals'],
    train_times=data['train_times'],
    holdout_times=data['holdout_times'],
    scaler=data['scaler'],
    device='cuda' if torch.cuda.is_available() else 'cpu',
)

trainer.train()

## 4. Visualize Results

In [ ]:
# Sample trajectories
model.eval()
x0 = data['marginals'][min(data['train_times'])][:200].to(trainer.device)

with torch.no_grad():
    trajectories, t_eval = model.sample(x0, n_steps=20, ema=True)

trajectories = trajectories.cpu().numpy()
t_eval = t_eval.cpu().numpy()

# Plot
fig, ax = plt.subplots(figsize=(12, 6))

# Plot trajectories
for i in range(trajectories.shape[1]):
    ax.plot(t_eval, trajectories[:, i, 0], alpha=0.3, lw=0.5, color='purple')

# Add violin plots
norm_times = [(t - min(all_times)) / (max(all_times) - min(all_times)) for t in all_times]
parts = ax.violinplot(pm25_data, positions=norm_times, showmeans=True, widths=0.05)

ax.set_xlabel('Normalized Time')
ax.set_ylabel('PM2.5 (normalized)')
ax.set_title('Learned PM2.5 Trajectories')
plt.show()

## Reproduce Paper Results

```bash
python experiments/train.py --dataset beijingair --config configs/beijing/defaults.json
```